# ResearchPilot: project walkthrough

This notebook runs the real ResearchPilot code locally. Open it in VS Code, select the `.venv` kernel, then choose **Run All**. The first run downloads the models; later runs use the cache. The Kaggle notebook is separate.

## 1. The problem

ResearchPilot answers questions over uploaded PDFs with a local Qwen model. A model can write a plausible answer, but it cannot show which page it came from. ResearchPilot retrieves the relevant passages first, makes Qwen answer only from those, and resolves citations back to stored chunks in code. If the evidence is weak, it abstains instead of guessing.


## 2. Architecture

```
POST /research → plan → search (web, or offline Demo corpus)  ┐
                      → document_search (uploaded PDFs)      ┴→ collect_sources → assess_evidence
assess_evidence → END                      (analyze = false)
                → report_writer → END       (not enough evidence)
                → analyst → verifier → report_writer → END
                     ↑_________|            (one revision at most)
```

Search and PDF retrieval run in parallel; `assess_evidence` decides whether there is enough evidence to analyze, and rejected findings can trigger one revision.

The Planner is deterministic Python. MiniLM handles retrieval embeddings, Qwen generates the Analyst findings and reviews claims, and LangGraph routes the stages.


## 3. Environment and device

The first cell finds the project root, so the notebook works if the kernel starts from a parent folder, and it warns if the kernel is not `.venv`. Setup is `uv sync --all-groups`; the first run needs the Hugging Face model download.

The device cell only selects CUDA, MPS, or CPU. It does not load the model.


In [ ]:
import os
import sys
from pathlib import Path


def find_project_root() -> Path:
    # Find the repository from the kernel's working directory.
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "app" / "rag" / "qa.py").exists() and (folder / "pyproject.toml").exists():
            return folder
    raise FileNotFoundError("Open this notebook from inside the researchpilot/ project folder.")


PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Python:      ", sys.executable)
if Path(sys.prefix).resolve() != (PROJECT_ROOT / ".venv").resolve():
    print("WARNING: this kernel is not the project's .venv.")
    print("In VS Code: Select Kernel → Python Environments → .venv")

In [ ]:
import importlib.util
from importlib.metadata import version

required = {
    "fastapi": "fastapi",
    "langgraph": "langgraph",
    "pypdf": "pypdf",
    "faiss": "faiss-cpu",
    "sentence_transformers": "sentence-transformers",
    "torch": "torch",
    "transformers": "transformers",
}
missing = [pkg for module, pkg in required.items() if importlib.util.find_spec(module) is None]
if missing:
    raise ImportError(f"Missing {', '.join(missing)}. Run `uv sync --all-groups`, then restart.")
print("  ".join(f"{pkg} {version(pkg)}" for pkg in required.values()))

In [ ]:
import torch

from app.services.llm import select_device

# Prefer CUDA, then Apple MPS, then CPU; float16 keeps the model near 3.1 GB.
device_config = select_device("auto")
DEVICE = device_config.device
print(f"CUDA: {torch.cuda.is_available()}   MPS: {torch.backends.mps.is_available()}")
print(f"Using {device_config.device} ({device_config.dtype}): {device_config.reason}")

## 4. Reading the PDF page by page

The sample is a synthetic five-page battery report, so a correct answer has to come from the PDF rather than the model's memory.

`load_pdf()` extracts text page by page, which preserves the page number for every later chunk and citation. The document ID is the SHA-256 hash of the file.


In [ ]:
import json

from IPython.display import Markdown, display

from app.rag.pdf import load_pdf

SAMPLE_DIR = PROJECT_ROOT / "sample_data" / "pdfs"
CASES = json.loads((SAMPLE_DIR / "researchpilot_test_report_cases.json").read_text())
SAMPLE_PDF = SAMPLE_DIR / CASES["pdf"]
document = load_pdf(SAMPLE_PDF)


def table(headers, rows):
    lines = ["| " + " | ".join(headers) + " |", "|" + "---|" * len(headers)]
    lines += [
        "| " + " | ".join(str(v).replace("|", "\\|").replace("\n", " ") for v in row) + " |"
        for row in rows
    ]
    display(Markdown("\n".join(lines)))


print(f"{document.filename}: {document.page_count} pages, document_id {document.document_id}")
table(
    ["Page", "Characters", "Start of page"],
    [(p.page_number, len(p.text), p.text[:90] + "…") for p in document.pages],
)

## 5. Chunks, embeddings, and FAISS

I split each page into chunks of about 500 characters with 100 characters of overlap. Chunks never cross pages, so each one keeps a page number; the ID `…:p2:c0` means page 2, chunk 0.

MiniLM embeds the chunks, and FAISS returns the closest matches to the question. The `0.25` minimum score drops weak matches before Qwen sees anything.


In [ ]:
from app.rag.chunking import chunk_document
from app.rag.embeddings import SentenceTransformerEmbeddings
from app.rag.retrieval import VectorRetriever
from app.rag.vector_store import FaissVectorStore

chunks = chunk_document(document)
embedder = SentenceTransformerEmbeddings()
vectors = embedder.embed_documents([c.text for c in chunks])
store = FaissVectorStore(embedder.dimension)
store.add(chunks, vectors)
retriever = VectorRetriever(embedder, store, min_score=0.25)

print(f"{len(chunks)} chunks, embedding matrix {vectors.shape}")
query = "Which battery option was selected and why?"
hits = store.search(embedder.embed_query(query), top_k=4)
print(f"Query: {query}")
table(
    ["Rank", "Similarity", "Page", "Chunk ID", "Text"],
    [
        (h.rank, f"{h.score:.3f}", h.chunk.page_number, h.chunk.chunk_id, h.chunk.text[:80] + "…")
        for h in hits
    ],
)

## 6. Running Qwen locally

`HuggingFaceLLM` loads Qwen on first use, so importing the project does not download the model. I use it without fine-tuning and with greedy decoding; if loading fails, the error is reported rather than replaced with fake output. The next section reuses this same model instance.

The tokenizer output is here because Qwen works on sub-word tokens, not whole words.


In [ ]:
import time

from app.services.llm import HuggingFaceLLM, LLMLoadError

llm = HuggingFaceLLM("Qwen/Qwen2.5-1.5B-Instruct", device=DEVICE)
started = time.perf_counter()
try:
    llm.load()
except LLMLoadError as exc:
    raise RuntimeError(
        f"{exc}\n\nThe first run downloads about 3.1 GB and needs Internet."
    ) from None
print(f"Loaded in {time.perf_counter() - started:.1f}s:", llm.describe())

ids = llm.tokenizer("Lithium iron phosphate batteries.")["input_ids"]
print("Tokens:", llm.tokenizer.convert_ids_to_tokens(ids))

## 7. Answering from the PDF, with checked citations

This is the standalone RAG path: `answer_question()` retrieves up to three chunks, calls Qwen at most once, and checks the reply directly. The full LangGraph workflow comes next.

The prompt labels the chunks `E1`, `E2`, and `E3`, and Qwen has to cite those IDs. Code maps each ID back to the stored chunk, so the filename and page never come from the model. Invented IDs are dropped, and the answer is withheld if no valid citation remains or none of its sentences match the cited text. If retrieval finds nothing above the threshold, it abstains without calling Qwen.


In [ ]:
from app.rag.qa import answer_question, render_markdown

answerable = next(c for c in CASES["cases"] if c["id"] == "efficiency")
unanswerable = next(c for c in CASES["cases"] if c["id"] == "unanswerable-manufacturer")

for case in (answerable, unanswerable):
    result = answer_question(case["question"], retriever, llm, top_k=3)
    display(Markdown(render_markdown(result)))
    print(f"Expected page(s): {case['expected_pages'] or 'none (not in the PDF)'}")
    print(f"Raw model output: {result.raw_output!r}\n" + "-" * 80)

The next cell uses a hand-written reply, not Qwen, so I can show `E9` being discarded while `E1` resolves to page 3.


In [ ]:
from app.services.llm import Generation


# Hand-written reply: E1 exists, E9 does not.
class HandWrittenReply:
    model_id = "hand-written reply (not a model)"

    def generate(self, messages, *, max_new_tokens=256):
        text = "ANSWER: The round-trip efficiency was 90.5%.\nEVIDENCE: E1, E9"
        return Generation(text, prompt_tokens=0, completion_tokens=0, duration_ms=0.0)


checked = answer_question(answerable["question"], retriever, HandWrittenReply())
print("Kept:", [(c.label, c.filename, c.page_number) for c in checked.citations])
print("Discarded:", checked.invalid_citations)

## 8. The LangGraph workflow, through the real API

This runs the real FastAPI app through `TestClient` with a temporary data folder. Demo mode is on for search, so no API key is needed, and the Qwen instance from above is reused. The printed graph shows the conditional routes.


In [ ]:
import tempfile

from fastapi.testclient import TestClient

from app.config import Settings
from app.main import create_app

settings = Settings(
    _env_file=None,
    data_dir=Path(tempfile.mkdtemp(prefix="researchpilot-walkthrough-")),
    llm_device=DEVICE,
    log_level="WARNING",  # keep request logs out of the output
)
app = create_app(settings, llm_factory=lambda: llm)
client = TestClient(app)
print("GET /health →", client.get("/health").json())

graph = app.state.research_service.workflow.get_graph()
for edge in graph.edges:
    print(f"  {edge.source:>16} {'⇢' if edge.conditional else '→'} {edge.target}")

In [ ]:
with open(SAMPLE_PDF, "rb") as handle:
    upload = client.post("/documents", files={"file": (SAMPLE_PDF.name, handle, "application/pdf")})
upload.raise_for_status()
DOCUMENT_ID = upload.json()["document_id"]  # taken from the response, never hard-coded
print(
    f"POST /documents → {upload.status_code}: {DOCUMENT_ID}, {upload.json()['chunk_count']} chunks"
)

### Analyst, Verifier, Report Writer

The Analyst gets the top sources and returns JSON findings with source IDs. The Verifier checks those IDs, page metadata, numbers, wording, subject, and relevance in code, then optionally asks Qwen to review each claim. `verified` needs both. Rejected findings can trigger one revision. The Report Writer formats the results with ordinary code and withholds the recommendation when a requested criterion has no verified finding.


### An answerable question

This is the same request the Streamlit UI uses. The output shows the source types, the analysis status, and the Analyst's JSON next to the Verifier's decisions.


In [ ]:
request = {**CASES["ui_demo_request"], "document_ids": [DOCUMENT_ID], "analyze": True}
started = time.perf_counter()
response = client.post("/research", json=request)
response.raise_for_status()
run = response.json()
print(f"Question: {run['request']['question']}")
print(
    f"POST /research → {response.status_code} in {time.perf_counter() - started:.1f}s; "
    f"analysis: {run['analysis']['status']}; LLM calls: {run['metrics']['llm_calls']}"
)

sources = {s["id"]: s for s in run["sources"]}


def where(source_id):
    s = sources[source_id]
    return (
        f"{s['filename']} p.{s['page_number']}"
        if s["source_type"] == "document"
        else f"{s['source_type']}: {s['url']}"
    )


table(
    ["ID", "Type", "Reference"],
    [(s["id"], s["source_type"], where(s["id"])) for s in run["sources"]],
)

In [ ]:
for attempt in run["analysis"]["attempts"]:
    print(f"Analyst attempt {attempt['attempt']}:\n{attempt['raw_output']}\n")

table(
    ["Status", "Claim", "Cites", "Why"],
    [
        (
            c["status"],
            c["claim"],
            "; ".join(f"{sid} ({where(sid)})" for sid in c["cited_source_ids"]),
            " ".join(c["explanations"]),
        )
        for c in run["analysis"]["verification"]["claims"]
    ],
)

### The stored report

The report is read back through the API and built from the verification results, not from the model's prose.


In [ ]:
stored = client.get(f"/research/{run['run_id']}").json()
print("Stored run identical to the response:", stored == run)
recommendation = stored["report"].split("## Recommendation")[1].split("## ")[0].strip()
print("Recommendation section:\n" + recommendation)
display(Markdown(stored["report"]))

## 9. A question the PDF cannot answer

The PDF never mentions electric vehicles, so the correct result is `insufficient_evidence`, with no verified findings. The relevance check matters here: during testing, a run cited a real sentence about a power outage that had nothing to do with the question.


In [ ]:
ev = next(c for c in CASES["cases"] if c["id"] == "unanswerable-ev")
response = client.post(
    "/research", json={"question": ev["question"], "document_ids": [DOCUMENT_ID], "analyze": True}
)
response.raise_for_status()
abstain = response.json()
print(f"Question: {abstain['request']['question']}\nanalysis: {abstain['analysis']['status']}")
for claim in (abstain["analysis"]["verification"] or {}).get("claims", []):
    print(f"  [{claim['status']}] {claim['claim']}  ← {claim['explanations'][-1]}")
summary = abstain["report"].split("## Executive summary")[1].split("## ")[0].strip()
display(Markdown("**Executive summary:** " + summary))

## 10. Automated tests

The test suite replaces Qwen with scripted replies and runs offline. It checks routing, schemas, citation and injection checks, revision limits, API behaviour, persistence, and the UI, but it does not measure model quality. The next cell runs pytest and Ruff.


In [ ]:
import re
import subprocess

for command in (["pytest", "-q", "--color=no", "-p", "no:cacheprovider"], ["ruff", "check", "."]):
    completed = subprocess.run(
        [sys.executable, "-m", *command],
        cwd=PROJECT_ROOT,
        capture_output=True,
        text=True,
        env={**os.environ, "NO_COLOR": "1"},
    )
    output = re.sub(r"\x1b\[[0-9;]*m", "", completed.stdout + completed.stderr).strip().splitlines()
    print(f"$ {' '.join(command)}  →  {output[-1]}  (exit code {completed.returncode})")

## 11. Limitations

- `verified` is a heuristic, not proof. The 92% rated versus 90.5% measured example shows how wording can match while the meaning is wrong.
- A correct paraphrase can remain `unverified`, and a relevant page can fall outside the top-k retrieval window.
- The 1.5B model is small; live Tavily, Supabase pgvector, and Kaggle were not tested in this release, and there is no OCR.

## 12. Questions I expect

**Why a 1.5B model?** It runs locally in about 3.1 GB; the verification layer exists because the model is small.

**How are citations checked?** IDs must map to sources in the run. Page numbers come from stored chunks, and invented IDs are dropped.

**Why not just word overlap?** Overlap cannot catch negation, so `verified` also needs the model review.

**Why LangGraph?** The routing, abstention, and revision are explicit and testable.


In [ ]:
client.close()

ResearchPilot shows the part I care about: the answer stays tied to stored PDF pages, unverified claims are labelled, and missing evidence leads to an abstention rather than a guess.
